## Step 1) Import libraries

In [1]:
from llama_index.llms.groq import Groq
from llama_index.embeddings.huggingface import HuggingFaceEmbedding
from llama_index.core import VectorStoreIndex
from llama_index.core.chat_engine import ContextChatEngine
from llama_index.core.memory import ChatMemoryBuffer
from llama_index.core.base.llms.types import ChatMessage, MessageRole
from llama_index.core.node_parser import SentenceSplitter

from dotenv import load_dotenv
import os
import gradio as gr

c:\Users\sanaz\miniconda3\envs\llama\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## Step 2) Load API keys

## Step 3) Set up the LLM

In [3]:
# Import Groq LLM from LlamaIndex
from llama_index.llms.groq import Groq

# Choose the LLM model
model = "llama-3.3-70b-versatile"

# Create the LLM object
llm = Groq(
    model=model,
    api_key=groq_api_key
)

## Step 4) Set up the embedding model

In [4]:
embedding_model_name = "sentence-transformers/all-MiniLM-L6-v2"

embeddings = HuggingFaceEmbedding(
    model_name=embedding_model_name,
    cache_folder="./embedding_model"
)

## Step 5) Load PDF documents

In [41]:
from llama_index.core import SimpleDirectoryReader

documents = SimpleDirectoryReader(
    "./data",
    required_exts=[".pdf"]
).load_data(show_progress=True)

print(f"Loaded {len(documents)} documents.")

Loading files: 100%|██████████| 1/1 [00:04<00:00,  4.50s/it]

Loaded 36 documents.


## Step 6) Split text into chunks

In [42]:

text_splitter = SentenceSplitter(
    chunk_size=800,
    chunk_overlap=150
)

## Step 7) Build vector index

In [43]:
# ===== Build vector index from your PDFs =====
vector_index = VectorStoreIndex.from_documents(
    documents,
    transformations=[text_splitter],
    embed_model=embeddings
)

## Step 8) Retriever 

In [44]:
retriever = vector_index.as_retriever(similarity_top_k=2)



## Step 9) prefix_messages

In [45]:
prefix_messages = [
    ChatMessage(
        role=MessageRole.SYSTEM,
        content="You are a nice chatbot having a conversation with a human.",
    ),
    ChatMessage(
        role=MessageRole.SYSTEM,
        content="Answer the question based only on the following context and previous conversation.",
    ),
    ChatMessage(
        role=MessageRole.SYSTEM,
        content='If the answer is not found in the document, say: "The answer is not found in the document."',
    ),
    ChatMessage(
        role=MessageRole.SYSTEM,
        content="Keep your answers short and succinct.",
    ),
]

## Step 10) Memory

In [46]:
memory = ChatMemoryBuffer.from_defaults()

## Step 11) RAG chatbot with memory

In [47]:
# ===== RAG chatbot with memory =====
rag_bot = ContextChatEngine(
    llm=llm,
    retriever=retriever,
    memory=memory,
    prefix_messages=prefix_messages,
)

## Step 12) Create a wrapper function for Gradio

In [48]:
import gradio as gr

def rag_bot_chat_interface(message, history):
    response = rag_bot.chat(message)
    return response.response

demo = gr.ChatInterface(
    fn=rag_bot_chat_interface,
    title="My RAG Chatbot",
    description="Ask questions about your PDF documents.",

)

demo.launch(share=True)

2026-03-30 15:12:22,881 - INFO - HTTP Request: GET http://127.0.0.1:7877/gradio_api/startup-events "HTTP/1.1 200 OK"
2026-03-30 15:12:22,895 - INFO - HTTP Request: HEAD http://127.0.0.1:7877/ "HTTP/1.1 200 OK"


* Running on local URL:  http://127.0.0.1:7877


2026-03-30 15:12:23,289 - INFO - HTTP Request: GET https://api.gradio.app/pkg-version "HTTP/1.1 200 OK"
2026-03-30 15:12:23,649 - INFO - HTTP Request: GET https://api.gradio.app/v3/tunnel-request "HTTP/1.1 200 OK"


* Running on public URL: https://4b99c2733e75d14ee4.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


2026-03-30 15:12:25,440 - INFO - HTTP Request: HEAD https://4b99c2733e75d14ee4.gradio.live "HTTP/1.1 200 OK"


## Step 13) Launch Gradio ChatInterface

In [49]:
#  Step 11) Gradio Web App 
demo = gr.ChatInterface(
    fn=rag_bot_chat_interface,
    title="RAG Chatbot with Memory",
    description="Ask questions about your PDF documents.",

)

demo.launch(share=True)

2026-03-30 15:12:26,007 - INFO - HTTP Request: GET http://127.0.0.1:7878/gradio_api/startup-events "HTTP/1.1 200 OK"
2026-03-30 15:12:26,024 - INFO - HTTP Request: HEAD http://127.0.0.1:7878/ "HTTP/1.1 200 OK"


* Running on local URL:  http://127.0.0.1:7878


2026-03-30 15:12:26,414 - INFO - HTTP Request: GET https://api.gradio.app/pkg-version "HTTP/1.1 200 OK"
2026-03-30 15:12:26,783 - INFO - HTTP Request: GET https://api.gradio.app/v3/tunnel-request "HTTP/1.1 200 OK"


* Running on public URL: https://2c09af14476ab99830.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


2026-03-30 15:12:28,556 - INFO - HTTP Request: HEAD https://2c09af14476ab99830.gradio.live "HTTP/1.1 200 OK"


## Step 14) Callback Function

In [50]:
def custom_rag_bot_callback(message, history, top_k_value):
    # Update retriever parameter (how many documents to use)
    rag_bot._retriever._similarity_top_k = int(top_k_value)

    # Reset shared memory to avoid mixing different users' chats
    rag_bot.reset()

    # Convert Gradio chat history to LlamaIndex ChatMessage format
    bot_history = [
    ChatMessage(
        role=m["role"],
        content=m["content"] if isinstance(m["content"], str) else m["content"][0]["text"]
    )
    for m in history
    ]

    # Get response from RAG chatbot using current message + chat history
    response = rag_bot.chat(message, chat_history=bot_history)

    # Append the new user message to history
    history.append({"role": "user", "content": message})

    # Append the bot response to history
    history.append({"role": "assistant", "content": response.response})

    # Return updated chat history to Gradio UI
    return history

## Step 15) Gradio UI (Blocks)

In [51]:
import gradio as gr

with gr.Blocks(title="Custom RAG Bot UI") as demo_custom:
    gr.Markdown("<h1>RAG Bot (Stateless/No-State Approach)</h1>")

    with gr.Row():
        with gr.Column(scale=1, min_width=250):
            gr.Markdown("## RAG Settings")
            top_k_slider = gr.Slider(
                minimum=1,
                maximum=5,
                step=1,
                value=2,
                label="Top K"
            )

        with gr.Column(scale=4):
            chatbot = gr.Chatbot(
                label="Climate change",
                height=500,
                value=[
                    {"role": "assistant", "content": "Hello! I'm here to help—what would you like to know?"}
                ]
            )

            msg_input = gr.Textbox(
                label="Your Question",
                placeholder="Ask me something..."
            )

            msg_input.submit(
                fn=custom_rag_bot_callback,
                inputs=[msg_input, chatbot, top_k_slider],
                outputs=[chatbot]
            ).then(
                fn=lambda: "",
                outputs=[msg_input]
            )

demo_custom.launch(theme="soft", share=True)

2026-03-30 15:12:29,087 - INFO - HTTP Request: GET http://127.0.0.1:7879/gradio_api/startup-events "HTTP/1.1 200 OK"
2026-03-30 15:12:29,100 - INFO - HTTP Request: HEAD http://127.0.0.1:7879/ "HTTP/1.1 200 OK"


* Running on local URL:  http://127.0.0.1:7879


2026-03-30 15:12:29,491 - INFO - HTTP Request: GET https://api.gradio.app/pkg-version "HTTP/1.1 200 OK"
2026-03-30 15:12:29,871 - INFO - HTTP Request: GET https://api.gradio.app/v3/tunnel-request "HTTP/1.1 200 OK"


* Running on public URL: https://2ec64b2f7ef7e936d5.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


2026-03-30 15:12:31,660 - INFO - HTTP Request: HEAD https://2ec64b2f7ef7e936d5.gradio.live "HTTP/1.1 200 OK"


In [ ]:
#  Import required libraries

import os
import gradio as gr

from dotenv import load_dotenv

from llama_index.core import VectorStoreIndex, Settings
from llama_index.core.llms import ChatMessage
from llama_index.llms.groq import Groq
from llama_index.embeddings.huggingface import HuggingFaceEmbedding
from llama_index.readers.file import PyMuPDFReader
from llama_index.core.node_parser import SentenceSplitter
 
# Load API key from .env
 
load_dotenv()

 
# LLM + Embedding setup

Settings.llm = Groq(model="llama-3.3-70b-versatile")
Settings.embed_model = HuggingFaceEmbedding(
    model_name="sentence-transformers/all-MiniLM-L6-v2"
)


# Global variables
rag_bot = None

# Theme and CSS
custom_theme = gr.themes.Soft(
    primary_hue="green",
    secondary_hue="blue",
    neutral_hue="gray",
    font=gr.themes.GoogleFont("Inter")
)

custom_css = """
.gradio-container {
    max-width: 1200px !important;
    margin: auto !important;
    padding: 20px !important;
}
.chatbot-container {
    border-radius: 15px;
    overflow: hidden;
    box-shadow: 0 4px 6px rgba(0, 0, 0, 0.1);
}
"""

# Control how the PDF is split into chunks
Settings.node_parser = SentenceSplitter(
    chunk_size=500,        # number of tokens per chunk
    chunk_overlap=50       # overlap between chunks
)


# Create RAG bot from uploaded PDF
def load_pdf_and_create_bot(pdf_file, top_k_value):
    global rag_bot

    if pdf_file is None:
        return "Please upload a PDF file first."

    try:
        loader = PyMuPDFReader()
        documents = loader.load_data(file_path=pdf_file)

        readable_docs = [
            doc for doc in documents
            if hasattr(doc, "text") and doc.text and doc.text.strip()
        ]

        if not readable_docs:
            return "The PDF was loaded, but no readable text was found."

        index = VectorStoreIndex.from_documents(readable_docs)

        rag_bot = index.as_chat_engine(
            chat_mode="context",
            similarity_top_k=int(top_k_value),
            verbose=True
        )
        system_prompt="""
You are a helpful and professional assistant.

Answer the user's question ONLY using the provided PDF context.

If the answer is not in the document, say: "I don't know based on the document."

Be clear, short, and accurate.
"""

        total_chars = sum(len(doc.text) for doc in readable_docs)

        return (
            f"PDF uploaded successfully. "
            f"Loaded {len(readable_docs)} document(s) with about {total_chars} characters."
        )

    except Exception as e:
        return f"Error while loading PDF: {e}"

# Chat callback
def custom_rag_bot_callback(message, history, top_k_value):
    global rag_bot

    history = history or []

    if not message.strip():
        return history

    if rag_bot is None:
        history.append({"role": "user", "content": message})
        history.append({
            "role": "assistant",
            "content": "Please upload and load a PDF first."
        })
        return history

    try:
        # Update retriever setting
        rag_bot._retriever._similarity_top_k = int(top_k_value)

        # Convert Gradio history to LlamaIndex ChatMessage format
        bot_history = []
        for m in history:
            if isinstance(m, dict) and "role" in m and "content" in m:
                bot_history.append(
                    ChatMessage(
                        role=m["role"],
                        content=str(m["content"])
                    )
                )

        # Ask the model
        response = rag_bot.chat(message, chat_history=bot_history)

        # Update chat history
        history.append({"role": "user", "content": message})
        history.append({"role": "assistant", "content": response.response})

        return history

    except Exception as e:
        history.append({"role": "user", "content": message})
        history.append({"role": "assistant", "content": f"Error: {e}"})
        return history

# Optional: clear chatbot
def clear_chat():
    return [
        {
            "role": "assistant",
            "content": "Hello! Please upload a PDF first."
        }
    ], ""

# Build UI
with gr.Blocks(theme=custom_theme, css=custom_css, title="Custom RAG Bot UI") as demo_custom:
    gr.HTML("""
    <div style="text-align: center; margin-bottom: 20px;">
        <h1>🤖 RAG Bot</h1>
        <p>Upload a PDF and ask questions about it.</p>
    </div>
    """)

    with gr.Row():
        with gr.Column(scale=1, min_width=260):
            gr.Markdown("## ⚙️ RAG Settings")

            pdf_input = gr.File(
                label="Upload PDF",
                file_types=[".pdf"],
                type="filepath"
            )

            top_k_slider = gr.Slider(
                minimum=1,
                maximum=5,
                step=1,
                value=2,
                label="Top K"
            )

            upload_btn = gr.Button("Load PDF")
            clear_btn = gr.Button("Clear Chat")

            upload_status = gr.Textbox(
                label="Status",
                interactive=False
            )

        with gr.Column(scale=4, elem_classes="chatbot-container"):
            chatbot = gr.Chatbot(
                label="Climate Change Expert",
                height=500,
                layout="bubble",
                value=[
                    {
                        "role": "assistant",
                        "content": "Hello! Please upload a PDF first."
                    }
                ]
            )

            msg_input = gr.Textbox(
                label="Your Question",
                placeholder="Ask me something about the uploaded PDF..."
            )

            upload_btn.click(
                fn=load_pdf_and_create_bot,
                inputs=[pdf_input, top_k_slider],
                outputs=[upload_status]
            )

            msg_input.submit(
                fn=custom_rag_bot_callback,
                inputs=[msg_input, chatbot, top_k_slider],
                outputs=[chatbot]
            ).then(
                fn=lambda: "",
                outputs=[msg_input]
            )

            clear_btn.click(
                fn=clear_chat,
                outputs=[chatbot, msg_input]
            )

demo_custom.launch(share=True)

2026-03-31 09:22:28,248 - INFO - Load pretrained SentenceTransformer: sentence-transformers/all-MiniLM-L6-v2
C:\Users\sanaz\AppData\Local\Temp\ipykernel_27392\5936485.py:161: UserWarning: The parameters have been moved from the Blocks constructor to the launch() method in Gradio 6.0: theme, css. Please pass these parameters to launch() instead.
  with gr.Blocks(theme=custom_theme, css=custom_css, title="Custom RAG Bot UI") as demo_custom:
2026-03-31 09:22:30,635 - INFO - HTTP Request: GET http://127.0.0.1:7867/gradio_api/startup-events "HTTP/1.1 200 OK"
2026-03-31 09:22:30,666 - INFO - HTTP Request: HEAD http://127.0.0.1:7867/ "HTTP/1.1 200 OK"


* Running on local URL:  http://127.0.0.1:7867


2026-03-31 09:22:31,155 - INFO - HTTP Request: GET https://api.gradio.app/pkg-version "HTTP/1.1 200 OK"
2026-03-31 09:22:31,447 - INFO - HTTP Request: GET https://api.gradio.app/v3/tunnel-request "HTTP/1.1 200 OK"


* Running on public URL: https://a4ca960ca17f0edfd2.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


2026-03-31 09:22:33,281 - INFO - HTTP Request: HEAD https://a4ca960ca17f0edfd2.gradio.live "HTTP/1.1 200 OK"


2026-03-31 09:23:14,204 - INFO - HTTP Request: POST https://api.groq.com/openai/v1/chat/completions "HTTP/1.1 200 OK"
2026-03-31 09:23:53,872 - INFO - HTTP Request: POST https://api.groq.com/openai/v1/chat/completions "HTTP/1.1 200 OK"


In [52]:
import gradio as gr
print(gr.__version__)

6.10.0


2026-03-30 15:24:47,791 - INFO - HTTP Request: POST https://api.groq.com/openai/v1/chat/completions "HTTP/1.1 200 OK"
2026-03-30 15:25:15,848 - INFO - HTTP Request: POST https://api.groq.com/openai/v1/chat/completions "HTTP/1.1 200 OK"
